#  All Supervised ML Algorithms
---
Covers: Linear Regression, Ridge, Lasso, Logistic Regression, KNN,
Naive Bayes, Decision Tree, Random Forest, Gradient Boosting,
XGBoost, LightGBM, SVM

Each algorithm explained with analogy + code on loan_default_dataset.csv

# Import Libraries

In [14]:
# ============================================================================
# 1. IMPORT LIBRARIES
# ============================================================================
import warnings
from collections import Counter
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

# Suppress warnings
warnings.filterwarnings("ignore")

# 2. Load Dataset

In [15]:
# ============================================================================
# 2. LOAD DATASET & CLEAN COLUMNS
# ============================================================================
df = pd.read_csv("loan_default_dataset.csv")

# Clean column names (removes hidden leading/trailing spaces)
df.columns = df.columns.str.strip()

TARGET_COL = "Status"  # Set your target column name

# Drop ID / identifier columns if present (they shouldn't be trained on)
id_cols = [col for col in df.columns if "id" in col.lower()]
if id_cols:
    df.drop(columns=id_cols, inplace=True)
    print(f"Dropped identifier column(s): {id_cols}")

Dropped identifier column(s): ['ID']


In [8]:
df.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


# 3. EDA

In [16]:
# ============================================================================
# 3. EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================================
print("Dataset Shape:", df.shape)
print(f"\nTarget Distribution ({TARGET_COL}):")
print(df[TARGET_COL].value_counts(dropna=False, normalize=True) * 100)

Dataset Shape: (148670, 33)

Target Distribution (Status):
Status
0    75.355485
1    24.644515
Name: proportion, dtype: float64


# 4. PREPROCESSING & FEATURE ENGINEERING

In [17]:
# ============================================================================
# 4. PREPROCESSING & FEATURE ENGINEERING
# ============================================================================

# 4.1 Ensure Target is clean (drop missing targets and cast to int)
df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(int)

# 4.2 Handle Missing Values
# Numerical features -> median
num_cols = df.select_dtypes(include=[np.number]).columns.drop(TARGET_COL)
for c in num_cols:
    df[c].fillna(df[c].median(), inplace=True)

# Categorical features -> mode
cat_cols = df.select_dtypes(include=["object"]).columns
for c in cat_cols:
    df[c].fillna(df[c].mode()[0], inplace=True)

# 4.3 Outlier Handling (IQR Capping)
for c in num_cols:
    Q1 = df[c].quantile(0.25)
    Q3 = df[c].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[c] = df[c].clip(lower_bound, upper_bound)

# 4.4 Encode Categorical Features cleanly
# One-hot encode all categorical columns with dtype=int (avoids boolean/object issues)
if len(cat_cols) > 0:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

# 5. FEATURE / TARGET SEPARATION & TRAIN-TEST SPLIT

In [18]:
# ============================================================================
# 5. TRAIN-TEST SPLIT
# ============================================================================
X = df.drop(TARGET_COL, axis=1)
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. FEATURE SCALING

In [19]:
# ============================================================================
# 6. FEATURE SCALING
# ============================================================================
scaler = StandardScaler()

X_train_s = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_s = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

# Fill any residual NaNs from constant zero-variance columns (StandardScaler safeguard)
X_train_s.fillna(0, inplace=True)
X_test_s.fillna(0, inplace=True)

,year,loan_amount,rate_of_interest,Interest_rate_spread,Upfront_charges,term,property_value,income,Credit_Score,LTV,...,age_45-54,age_55-64,age_65-74,age_<25,age_>74,submission_of_application_to_inst,Region_North-East,Region_central,Region_south,Security_Type_direct
59366,0.0,1.767020,-0.528219,-0.879272,-0.846719,0.0,2.589006,1.749374,-0.896481,-1.325045,...,-0.551590,-0.530071,-0.401470,-0.095861,-0.225289,-1.346374,-0.092499,-0.249216,1.150204,0.013908
1052,0.0,-0.421140,1.054305,0.718672,-0.231356,0.0,-0.710483,0.449802,1.262362,0.885757,...,-0.551590,-0.530071,-0.401470,-0.095861,-0.225289,0.742736,-0.092499,-0.249216,-0.869411,0.013908
59663,0.0,-0.302861,-0.754293,-0.499970,2.098554,0.0,0.451837,-0.753506,-1.483687,-1.414202,...,-0.551590,-0.530071,-0.401470,-0.095861,4.438753,0.742736,-0.092499,-0.249216,1.150204,0.013908
70209,0.0,0.584231,0.602155,0.505327,0.578154,0.0,0.226872,-0.063609,-0.896481,0.360116,...,-0.551590,-0.530071,-0.401470,-0.095861,-0.225289,0.742736,-0.092499,-0.249216,-0.869411,0.013908
126566,0.0,-0.835116,0.828230,0.175558,-0.542648,0.0,-0.935448,-1.042299,-1.190084,0.495624,...,-0.551590,-0.530071,-0.401470,-0.095861,4.438753,0.742736,-0.092499,-0.249216,-0.869411,0.013908
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106197,0.0,-1.308231,-0.094155,1.997964,0.004919,0.0,-0.560506,-1.010211,-1.423239,-2.153622,...,-0.551590,-0.530071,2.490846,-0.095861,-0.225289,0.742736,-0.092499,-0.249216,1.150204,0.013908
42823,0.0,-1.071674,0.828230,1.623537,-0.060104,0.0,-1.085424,-0.929991,0.986030,0.287636,...,1.812941,-0.530071,-0.401470,-0.095861,-0.225289,0.742736,-0.092499,-0.249216,1.150204,0.013908
78610,0.0,0.820789,-1.658593,-2.146863,-1.073104,0.0,0.714297,0.578155,1.573235,-0.146856,...,1.812941,-0.530071,-0.401470,-0.095861,-0.225289,-1.346374,-0.092499,-0.249216,1.150204,0.013908
18681,0.0,-0.894255,2.184679,1.910597,0.684355,0.0,-0.897954,0.610243,1.452340,0.085387,...,-0.551590,-0.530071,-0.401470,-0.095861,-0.225289,-1.346374,-0.092499,-0.249216,-0.869411,0.013908


# 7. HANDLING CLASS IMBALANCE (SMOTE)

In [20]:
# ============================================================================
# 7. HANDLING CLASS IMBALANCE (SMOTE)
# ============================================================================
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_s, y_train)

# 8. FINAL STATUS CHECK

In [21]:
# ============================================================================
# 8. FINAL STATUS CHECK
# ============================================================================
print("\n" + "=" * 50)
print(f"✅ Data ready successfully!")
print(f"• Features: {X_train_bal.shape[1]}")
print(f"• Balanced Train Set: {len(X_train_bal)} rows")
print(f"• Test Set: {len(X_test_s)} rows")
print(f"• Class Distribution after SMOTE: {Counter(y_train_bal)}")
print("=" * 50)


✅ Data ready successfully!
• Features: 49
• Balanced Train Set: 179250 rows
• Test Set: 29734 rows
• Class Distribution after SMOTE: Counter({0: 89625, 1: 89625})


In [22]:
# ============================================================================
# EVALUATION FUNCTION
# ============================================================================
def eval_model(model, Xtr, ytr, Xte, yte, name):
    model.fit(Xtr, ytr)
    yp = model.predict(Xte)
    yprob = model.predict_proba(Xte)[:, 1] if hasattr(model, 'predict_proba') else yp
    acc = accuracy_score(yte, yp)
    prec = precision_score(yte, yp, zero_division=0)
    rec = recall_score(yte, yp, zero_division=0)
    f1 = f1_score(yte, yp, zero_division=0)
    try: auc = roc_auc_score(yte, yprob)
    except: auc = 0
    print(f"\n{'='*55}\n🤖 {name}\n{'='*55}")
    print(f'  Accuracy={acc:.4f} | Precision={prec:.4f} | Recall={rec:.4f}')
    print(f'  F1={f1:.4f} | AUC={auc:.4f}')
    cm = confusion_matrix(yte, yp)
    print(f'  CM: TN={cm[0][0]} FP={cm[0][1]} FN={cm[1][0]} TP={cm[1][1]}')
    return {'Model': name, 'Accuracy': round(acc,4), 'Precision': round(prec,4),
            'Recall': round(rec,4), 'F1': round(f1,4), 'AUC': round(auc,4)}

results = []

## 📈 1. Logistic Regression
**Analogy:** Draws a line, uses Sigmoid to give probability 0-1.
If prob >= 0.5 → Yes, else → No.
**When:** Binary classification, need probabilities, interpretable baseline.

In [23]:
results.append(eval_model(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    X_train_bal, y_train_bal, X_test_s, y_test, '1. Logistic Regression'
))


🤖 1. Logistic Regression
  Accuracy=0.8368 | Precision=0.6607 | Recall=0.6945
  F1=0.6771 | AUC=0.8503
  CM: TN=19792 FP=2614 FN=2239 TP=5089


## 🔵 2. KNN (K-Nearest Neighbors)
**Analogy:** "Tell me your friends, I'll tell you who you are!"
Look at K closest points → majority vote.
**When:** Small data, simple baseline. Scaling is MUST!

In [24]:
results.append(eval_model(
    KNeighborsClassifier(n_neighbors=5),
    X_train_bal, y_train_bal, X_test_s, y_test, '2. KNN'
))


🤖 2. KNN
  Accuracy=0.8095 | Precision=0.5795 | Recall=0.8283
  F1=0.6819 | AUC=0.8898
  CM: TN=18001 FP=4405 FN=1258 TP=6070


## 📊 3. Naive Bayes
**Analogy:** Assumes features are independent (naive!) but works well.
Uses Bayes theorem: P(A|B) = P(B|A)*P(A)/P(B)
**When:** Text classification, fast, small data.

In [25]:
results.append(eval_model(
    GaussianNB(),
    X_train_bal, y_train_bal, X_test_s, y_test, '3. Naive Bayes'
))


🤖 3. Naive Bayes
  Accuracy=0.8719 | Precision=1.0000 | Recall=0.4801
  F1=0.6487 | AUC=1.0000
  CM: TN=22406 FP=0 FN=3810 TP=3518


## 🌳 4. Decision Tree
**Analogy:** Game of 20 Questions! Yes/No splits to narrow down answer.
**When:** Interpretable, no scaling needed. **Caution:** Overfits easily!

In [26]:
results.append(eval_model(
    DecisionTreeClassifier(max_depth=5, min_samples_split=20,
                           min_samples_leaf=10, class_weight='balanced', random_state=42),
    X_train_bal, y_train_bal, X_test_s, y_test, '4. Decision Tree'
))


🤖 4. Decision Tree
  Accuracy=1.0000 | Precision=1.0000 | Recall=1.0000
  F1=1.0000 | AUC=1.0000
  CM: TN=22406 FP=0 FN=0 TP=7328


## 🌲 5. Random Forest
**Analogy:** Ask 100 people → majority vote! Wisdom of crowds.
Each tree gets random data + random features (BAGGING).
**When:** Powerful, robust, feature importance.

In [27]:
results.append(eval_model(
    RandomForestClassifier(n_estimators=200, max_depth=10,
                           class_weight='balanced', random_state=42, n_jobs=-1),
    X_train_bal, y_train_bal, X_test_s, y_test, '5. Random Forest'
))


🤖 5. Random Forest
  Accuracy=1.0000 | Precision=1.0000 | Recall=1.0000
  F1=1.0000 | AUC=1.0000
  CM: TN=22406 FP=0 FN=0 TP=7328


## 🚀 6. Gradient Boosting
**Analogy:** Student gets 60/100. Tutor fixes mistakes. Next tutor fixes remaining.
Trees built SEQUENTIALLY, each fixing previous errors.
**When:** High accuracy needed.

In [28]:
results.append(eval_model(
    GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                               max_depth=5, subsample=0.8, random_state=42),
    X_train_bal, y_train_bal, X_test_s, y_test, '6. Gradient Boosting'
))


🤖 6. Gradient Boosting
  Accuracy=1.0000 | Precision=1.0000 | Recall=1.0000
  F1=1.0000 | AUC=1.0000
  CM: TN=22406 FP=0 FN=0 TP=7328


## 🚀 7. XGBoost
**Analogy:** Gradient Boosting's SUPERHERO — faster, regularized.
**When:** Best accuracy, handles missing values, Kaggle winner.

In [ ]:
results.append(eval_model(
    XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                  subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=1.0,
                  scale_pos_weight=len(y_train[y_train==0])/max(len(y_train[y_train==1]),1),
                  random_state=42, use_label_encoder=False, eval_metric='logloss'),
    X_train_bal, y_train_bal, X_test_s, y_test, '7. XGBoost'
))

## 🚀 8. LightGBM
**Analogy:** XGBoost's faster sibling. Leaf-wise growth.
**When:** Large datasets, fastest boosting.

In [ ]:
results.append(eval_model(
    LGBMClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                   is_unbalance=True, random_state=42, verbose=-1),
    X_train_bal, y_train_bal, X_test_s, y_test, '8. LightGBM'
))

## 📏 9. SVM (Support Vector Machine)
**Analogy:** Draw widest road between groups. Kernel trick maps to higher dimensions.
**When:** High-dimensional data. **Caution:** Slow on large data!

In [ ]:
results.append(eval_model(
    SVC(kernel='rbf', C=1.0, gamma='scale', probability=True,
        class_weight='balanced', random_state=42),
    X_train_bal, y_train_bal, X_test_s, y_test, '9. SVM (RBF)'
))

In [ ]:
# ============================================================================
# REGRESSION DEMO: Linear, Ridge, Lasso
# ============================================================================
print('\n' + '='*55)
print('📈 REGRESSION DEMO')
print('='*55)

reg_col = next((c for c in ['Income','LoanAmount','CreditScore'] if c in df.columns), None)
if reg_col:
    feats = [c for c in df.select_dtypes(include=[np.number]).columns if c not in ['Default', reg_col]]
    Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(df[feats], df[reg_col], test_size=0.2, random_state=42)
    rs = StandardScaler()
    Xr_tr_s, Xr_te_s = rs.fit_transform(Xr_tr), rs.transform(Xr_te)
    for name, m in [('Linear', LinearRegression()), ('Ridge L2', Ridge(1.0)), ('Lasso L1', Lasso(0.1))]:
        m.fit(Xr_tr_s, yr_tr)
        p = m.predict(Xr_te_s)
        print(f'\n{name}: MAE={mean_absolute_error(yr_te,p):.2f} RMSE={np.sqrt(mean_squared_error(yr_te,p)):.2f} R2={r2_score(yr_te,p):.4f}')
        if name == 'Lasso L1':
            print(f'  → Lasso eliminated {sum(m.coef_==0)} features!')
else:
    print('No suitable regression column found.')

In [ ]:
# ============================================================================
# FINAL COMPARISON
# ============================================================================
res_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
res_df.index += 1
print('\n🏆 ALL MODELS COMPARISON:')
print(res_df)

fig, ax = plt.subplots(figsize=(10, 6))
sorted_r = res_df.sort_values('F1')
ax.barh(sorted_r['Model'], sorted_r['F1'], color='steelblue', edgecolor='black')
for i, v in enumerate(sorted_r['F1']):
    ax.text(v+0.005, i, f'{v:.4f}', va='center', fontweight='bold')
ax.set_xlabel('F1 Score')
ax.set_title('All Supervised Models - F1 Score Comparison', fontweight='bold')
plt.tight_layout()
plt.show()